# CR Impact Analyzer
## Analyze a Change Request


### Configuration

In [ ]:
import os

BASE_DIR         = os.getcwd()
VECTOR_STORE_DIR = os.path.join(BASE_DIR, "vector_store")
COLLECTION_NAME  = "system_docs"
EMBEDDING_MODEL  = "all-mpnet-base-v2"   # must match Notebook 2
OLLAMA_BASE_URL  = "http://localhost:11434"

LLM_MODEL = "gemma:7b"

TOP_K_TARGETED = 6   # chunks from the SAME CR as the query
TOP_K_GENERAL  = 4   # additional chunks from OTHER related CRs
# Total context sent to LLM = 10 chunks max (was 25 — much less noise)

print("Config ready")
print(f"   LLM model        : {LLM_MODEL}")
print(f"   Targeted chunks  : {TOP_K_TARGETED}")
print(f"   General chunks   : {TOP_K_GENERAL}")

###  Step 1 Load the vector store

In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer

print("Loading vector store...")
client      = chromadb.PersistentClient(path=VECTOR_STORE_DIR)
collection  = client.get_collection(COLLECTION_NAME)
embed_model = SentenceTransformer(EMBEDDING_MODEL)
print(f" {collection.count()} chunks loaded")
print(" Embedding model ready")

### Step 2 — Enter your CR details here


In [ ]:
CR_NUMBER = "CR_Name"

CR_DESCRIPTION = """
CR_Description
"""


### Step 3 — Smart Retrieval (targeted + general)

In [ ]:
import time

def retrieve_targeted(query, cr_number, top_k):
    """Retrieve chunks specifically from the given CR number."""
    q_emb = embed_model.encode([query]).tolist()
    try:
        results = collection.query(
            query_embeddings=q_emb,
            n_results=top_k,
            where={"cr_number": cr_number},
            include=["documents", "metadatas", "distances"],
        )
        return [
            {
                "text":      doc,
                "source":    meta["source"],
                "cr_number": meta.get("cr_number", "?"),
                "relevance": round(1 - dist, 4),
                "type":      "targeted"
            }
            for doc, meta, dist in zip(
                results["documents"][0],
                results["metadatas"][0],
                results["distances"][0],
            )
        ]
    except Exception as e:
        print(f"    Targeted retrieval failed: {e}")
        return []


def retrieve_general(query, top_k, exclude_cr=None):
    """Retrieve chunks from the full knowledge base."""
    q_emb = embed_model.encode([query]).tolist()
    results = collection.query(
        query_embeddings=q_emb,
        n_results=top_k + 10,  # fetch extra to filter out excluded CR
        include=["documents", "metadatas", "distances"],
    )
    hits = []
    for doc, meta, dist in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0],
    ):
        if exclude_cr and meta.get("cr_number") == exclude_cr:
            continue  # skip — already covered by targeted retrieval
        hits.append({
            "text":      doc,
            "source":    meta["source"],
            "cr_number": meta.get("cr_number", "?"),
            "relevance": round(1 - dist, 4),
            "type":      "general"
        })
        if len(hits) >= top_k:
            break
    return hits


print(f"Retrieving context for {CR_NUMBER}...\n")
t0 = time.time()

# 1. Get chunks from the specific CR
targeted_hits = retrieve_targeted(CR_DESCRIPTION, CR_NUMBER, TOP_K_TARGETED)
print(f"    Targeted ({CR_NUMBER}): {len(targeted_hits)} chunks")

# 2. Get related chunks from other CRs
general_hits = retrieve_general(CR_DESCRIPTION, TOP_K_GENERAL, exclude_cr=CR_NUMBER)
print(f"    General (other CRs) : {len(general_hits)} chunks")

# 3. Combine and sort by relevance (best first)
all_hits = sorted(targeted_hits + general_hits, key=lambda x: x["relevance"], reverse=True)

print(f"\n   Retrieved {len(all_hits)} total chunks in {time.time()-t0:.2f}s\n")
print(f"{'#':<4} {'Type':<10} {'Relevance':<12} {'CR':<15} Source")
print("-" * 75)
for i, h in enumerate(all_hits, 1):
    bar = "█" * int(h['relevance'] * 10) + "░" * (10 - int(h['relevance'] * 10))
    src = h['source'][:35]
    print(f"{i:<4} {h['type']:<10} {bar} {h['relevance']:.1%}  {h['cr_number']:<15} {src}")

###  Step 4  Preview any retrieved chunk *(optional)*

In [ ]:
PREVIEW_INDEX = 0   # change to 0,1,2... to preview different chunks

h = all_hits[PREVIEW_INDEX]
print(f"Chunk [{PREVIEW_INDEX+1}]  |  {h['source']}  |  {h['cr_number']}  |  relevance {h['relevance']:.1%}")
print("=" * 70)
print(h["text"])

### Step 5 — Generate the CR Impact Analysis

In [ ]:
import requests
import json

context_parts = []
for i, h in enumerate(all_hits, 1):
    context_parts.append(
        f"[Excerpt {i} | Source: {h['source']} | CR: {h['cr_number']} | Relevance: {h['relevance']:.1%}]\n"
        f"{h['text']}"
    )
CONTEXT = "\n\n" + ("\n\n" + "-"*60 + "\n\n").join(context_parts)

PROMPT = f"""You are a senior software architect and order-management systems analyst.
Your task is to produce a structured impact analysis for the Change Request described below.

RULES:
- Use ONLY information from the RETRIEVED DOCUMENTATION below.
- Reference specific component names, APIs, modules, and workflows from the docs.
- Do NOT invent components or technologies not mentioned in the docs.
- Be specific and technical. Avoid vague statements.

CHANGE REQUEST: {CR_NUMBER}
{CR_DESCRIPTION.strip()}


RETRIEVED DOCUMENTATION:

{CONTEXT}


Generate the analysis using EXACTLY this structure:


## 1. Overview
- CR Name and Number
- Main Objective
- Summary of functional changes

## 2. Affected Components
List every impacted module, service, API, database, UI, or integration.
For each: Component Name | Type | How it is impacted

## 3. Workflow Changes
Describe all new or modified flows including:
- New statuses or states
- Retry and escalation logic
- Automatic vs manual actions
- State transitions

## 4. Dependencies
- Internal system dependencies
- External API dependencies
- Configuration dependencies (configurable values, SR-changeable params)
- Preconditions that must be met

## 5. Risks
For each risk: Risk | Level (Low/Medium/High/Critical) | Mitigation

## 6. Testing Requirements
List the critical scenarios that MUST be tested based on the documentation.

## 7. Overall Assessment
- Implementation complexity: Low / Medium / High
- Overall risk level: Low / Medium / High / Critical
- Most critical dependency
- Final recommendation
"""

print(f"Generating analysis using {LLM_MODEL}... (streaming)\n")
print("=" * 70)

collected = []
payload   = {"model": LLM_MODEL, "prompt": PROMPT, "stream": True}

try:
    with requests.post(
        f"{OLLAMA_BASE_URL}/api/generate",
        json=payload, stream=True, timeout=600
    ) as resp:
        resp.raise_for_status()
        for line in resp.iter_lines():
            if line:
                chunk = json.loads(line)
                token = chunk.get("response", "")
                print(token, end="", flush=True)
                collected.append(token)
                if chunk.get("done"):
                    break

    FULL_ANALYSIS = "".join(collected)
    print("\n" + "=" * 70)
    print("\n Analysis complete!")

except requests.exceptions.ConnectionError:
    print(" Cannot reach Ollama — make sure it is running in the background.")
except Exception as e:
    print(f" Error: {e}")

### Step 6 — Save the analysis

In [ ]:
import datetime

output_dir = os.path.join(BASE_DIR, "analyses")
os.makedirs(output_dir, exist_ok=True)

timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
filename  = os.path.join(output_dir, f"{CR_NUMBER}_{timestamp}.txt")

with open(filename, "w", encoding="utf-8") as f:
    f.write(f"CR: {CR_NUMBER}\n")
    f.write(f"{'='*60}\n")
    f.write(CR_DESCRIPTION.strip())
    f.write(f"\n\nSOURCES USED\n{'='*60}\n")
    for h in all_hits:
        f.write(f"  • [{h['type']}] {h['source']}  |  {h['cr_number']}  |  relevance {h['relevance']:.1%}\n")
    f.write(f"\n\nANALYSIS\n{'='*60}\n")
    f.write(FULL_ANALYSIS)

print(f" Saved to: {filename}")